# DiffuGroundingDINO trên Kaggle 2×T4

Finetune `groundingdino_swint_ogc.pth` với nhánh diffusion trên reference point (xem `object-detection/diffu_grounding_dino/README.md`). Notebook này dùng **cả 2 GPU T4** qua `torchrun` để tăng tốc, khác với `tools/run_train.py` (dùng cho GPU server riêng, chỉ 1 GPU mỗi lần).

**Trước khi chạy:**
1. Settings > Accelerator: **GPU T4 x2**. Settings > Internet: **On** (để `git clone`).
2. Add Data — Kaggle Dataset `objdet` (owner `cryandrrich`) chứa 2 file zip đã chuẩn bị sẵn từ máy dev:
   - `diffu_grounding_dino_weights.zip` (groundingdino_swint_ogc.pth + bert-base-uncased, ~1GB)
   - `data_coco_for_diffu_gdino.zip` (coco + coco_minitrain, ~4,6GB)

   **Kaggle tự giải nén mỗi zip vào một thư mục cùng tên (bỏ đuôi `.zip`) khi add dataset** — notebook này đọc thẳng từ thư mục đã giải nén sẵn ở `/kaggle/input/...`, không tự unzip nữa (unzip lại vào `/kaggle/working` vừa thừa vừa tốn thời gian/disk).
3. Code lấy trực tiếp từ GitHub (`git clone`) chứ không cần đóng gói riêng — sửa `REPO_URL`/`BRANCH` ở cell dưới nếu khác.

In [ ]:
import os

REPO_URL = "https://github.com/CryAndRRich/object-detection.git"
BRANCH = "main"

USE_DIFFUSION = True
EPOCHS = 15             # KHÔNG đổi giữa các phiên nối tiếp -- LR schedule (StepLR) và
                        # start_epoch tính theo tổng số epoch này; đổi giữa chừng làm
                        # lệch lịch LR so với phiên trước.
# batch=4/GPU đo được ~8.1GB/16GB (max mem: 8118 MiB) -- còn dư nhiều. Tăng lên 6 (không
# tăng gấp đôi lên 8) để chừa headroom cho ảnh COCO cỡ lớn hơn mức đã thấy trong 30 bước
# đầu + bộ nhớ tạm lúc eval, tránh OOM giữa 1 phiên Save & Run All (OOM giữa chừng phí
# gần hết 12h vì mất công clone/giải nén lại từ đầu phiên sau).
BATCH_SIZE = 6          # mỗi GPU -- torchrun chạy 2 tiến trình, tổng batch = 2x giá trị này
LR = 1e-4
SAMPLING_STEPS = 3      # số bước DDIM lúc eval
NUM_WORKERS = 2
RUN_TESTS = True

# ---- Dừng sạch sau N epoch mỗi phiên, thay vì để Kaggle giết tiến trình khi hết giờ ----
# Mỗi epoch ~5-6h (25.000 ảnh) -- 1 phiên Save & Run All chỉ nên ôm 1 epoch để chắc chắn
# xong TRƯỚC khi hết giờ (~9-12h, trừ hao ~15-20 phút setup/verify đầu phiên). Nếu để
# main.py tự chạy thẳng tới EPOCHS=15 và bị Kaggle cắt ngang giữa epoch: epoch dở dang đó
# mất trắng (checkpoint chỉ lưu sau khi 1 epoch train+eval xong), và các cell phía sau
# (đọc log, eval sweep, dọn dẹp) không bao giờ chạy tới -- không có cách nào biết kết quả
# ngay trong phiên đó. Dừng chủ động ở đúng ranh giới 1 epoch giải quyết cả hai: luôn có
# checkpoint đầy đủ của epoch vừa xong, và mọi cell sau LUÔN được chạy mỗi phiên.
# Đếm từ start_epoch của phiên này (main.py --max_epochs_this_run), không phải số epoch
# tuyệt đối -- nên giữ nguyên = 1 ở mọi phiên, không cần biết resume đang ở epoch nào.
MAX_EPOCHS_THIS_RUN = 1

# ---- Resume qua nhiều phiên Kaggle (mỗi phiên Save & Run All chỉ chạy được <=12h) ----
# Để trống ở phiên đầu tiên (train từ pretrain checkpoint). Từ phiên thứ 2 trở đi: sau khi
# phiên trước dừng sạch (đúng lúc MAX_EPOCHS_THIS_RUN, không phải bị Kaggle cắt), vào tab
# Output của notebook, thêm chính output đó làm 1 input data source mới (Add Data >
# Notebook Output > chọn version vừa chạy), rồi set RESUME trỏ vào checkpoint.pth trong
# input đó, chạy lại Save & Run All. main.py tự đọc epoch đã lưu trong checkpoint và tiếp
# tục đúng chỗ (không lặp lại epoch đã qua, không reset LR schedule).
RESUME = ""             # vd: "/kaggle/input/<ten-notebook-output>/output/diffu_run1/checkpoint.pth"

# Chỉ bật ở phiên CUỐI (khi đã đủ EPOCHS) -- eval sweep quét 4 mức sampling step, mỗi mức
# chạy full pass qua 5000 ảnh val, khá tốn thời gian. Bật ở phiên giữa chừng chỉ ăn vào
# quỹ 12h mà lẽ ra nên dành cho train thêm epoch.
FINAL_EVAL_SWEEP = False

WORK = "/kaggle/working"
REPO = f"{WORK}/object-detection"
CODE = f"{REPO}/diffu_grounding_dino"

# Kaggle tự giải nén mỗi zip vào thư mục cùng tên (bỏ .zip) ngay dưới dataset input --
# đổi lại nếu tên dataset/zip của bạn khác.
WEIGHTS_INPUT_DIR = "/kaggle/input/datasets/cryandrrich/objdet/diffu_grounding_dino_weights"
DATA_INPUT_DIR = "/kaggle/input/datasets/cryandrrich/objdet/data_coco_for_diffu_gdino"

OUT_DIR = f"{WORK}/output/diffu_run1"

## 1. Kiểm tra môi trường + cài lib

torch/torchvision Kaggle đã có sẵn khớp CUDA của máy -- **không cài lại**, chỉ thêm 3 lib nhẹ project cần (đúng `requirements.txt`, trừ torch/torchvision).

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| GPU count:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, (
    f"cần 2 GPU T4 (Settings > Accelerator > GPU T4 x2), đang thấy {torch.cuda.device_count()}"
)

!pip install -q 'transformers>=4.30,<5' 'scipy>=1.9' 'pycocotools>=2.0.6'

## 2. Lấy code từ GitHub

In [ ]:
import subprocess

if not os.path.isdir(REPO):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO],
        check=True,
    )
os.chdir(CODE)
print("cwd:", os.getcwd())

## 3. Trỏ thẳng vào weights + data (Kaggle đã tự giải nén khi add dataset)

In [ ]:
PRETRAIN = f"{WEIGHTS_INPUT_DIR}/diffu_grounding_dino/groundingdino_swint_ogc.pth"
BERT_DIR = f"{WEIGHTS_INPUT_DIR}/diffu_grounding_dino/bert-base-uncased"
assert os.path.exists(PRETRAIN), PRETRAIN
assert os.path.isdir(BERT_DIR), BERT_DIR
print("weights OK:", PRETRAIN, BERT_DIR)

In [ ]:
TRAIN_ANN = f"{DATA_INPUT_DIR}/coco_minitrain/annotations/instances_minitrain2017.json"
TRAIN_IMAGES = f"{DATA_INPUT_DIR}/coco_minitrain/images/train2017"
VAL_ANN = f"{DATA_INPUT_DIR}/coco/annotations/instances_val2017.json"
VAL_IMAGES = f"{DATA_INPUT_DIR}/coco/val2017"
for p in (TRAIN_ANN, TRAIN_IMAGES, VAL_ANN, VAL_IMAGES):
    assert os.path.exists(p), p
print("data OK")

## 4. Convert COCO-minitrain sang ODVG + dựng datasets json

In [ ]:
ANN_DIR = f"{WORK}/annotations"
os.makedirs(ANN_DIR, exist_ok=True)
ODVG_JSONL = f"{ANN_DIR}/minitrain_odvg.jsonl"
LABEL_MAP = f"{ANN_DIR}/label_map.json"

!python tools/coco2odvg.py --input {TRAIN_ANN} --output-jsonl {ODVG_JSONL} --output-label-map {LABEL_MAP}

In [ ]:
import json

DATASETS_JSON = f"{WORK}/datasets.json"
datasets_cfg = {
    "train": [{"root": TRAIN_IMAGES, "anno": ODVG_JSONL, "label_map": LABEL_MAP, "dataset_mode": "odvg"}],
    "val": [{"root": VAL_IMAGES, "anno": VAL_ANN, "dataset_mode": "coco"}],
}
with open(DATASETS_JSON, "w") as f:
    json.dump(datasets_cfg, f)

CONFIG = "config/cfg_odvg_diffusion.py" if USE_DIFFUSION else "config/cfg_odvg.py"
# NUM_WORKERS is NOT here -- main.py already exposes it as its own --num_workers CLI flag,
# so passing it through --options as well raises "config key collides with a command-line
# argument" (load_config_into_args rejects any --options key that shadows a real CLI arg).
OPTIONS = [
    f"text_encoder_type={BERT_DIR!r}",
    f"epochs={EPOCHS}",
    f"batch_size={BATCH_SIZE}",
    f"lr={LR}",
]
if USE_DIFFUSION:
    OPTIONS.append(f"diff_sampling_timesteps={SAMPLING_STEPS}")
opt_str = " ".join(OPTIONS)
print(DATASETS_JSON, "\n", CONFIG, "\n", OPTIONS)

## 5. Verification (80 test CPU + checkpoint key-compat)

Chạy trước khi train thật -- rẻ, bắt được lỗi wiring trước khi tốn GPU-hour. Xem README mục "Verification" để biết 8 bước tương ứng.

In [ ]:
if RUN_TESTS:
    !python tests/run_all.py

In [ ]:
!python tools/check_checkpoint.py -c {CONFIG} --checkpoint {PRETRAIN} --options {opt_str}

## 6. Train — **2 GPU song song qua `torchrun`**

`--nproc_per_node=2` chạy 2 tiến trình, mỗi tiến trình 1 GPU, `DistributedDataParallel` tự đồng bộ gradient sau mỗi bước (đã verify cơ chế này đúng bằng `tests/test_ddp.py` trước khi đưa lên đây). `BATCH_SIZE` ở trên là **mỗi GPU** -- tổng batch thực tế train là `2 * BATCH_SIZE`.

**Về thời gian và nhiều phiên:** mỗi epoch (25.000 ảnh) mất ~5-6 giờ, vượt quá 1 phiên Save & Run All (Kaggle giới hạn ~9-12h/phiên) nếu chạy thẳng `EPOCHS=15`. Notebook này dừng sạch sau `MAX_EPOCHS_THIS_RUN` epoch mỗi phiên (`main.py --max_epochs_this_run`) thay vì để Kaggle cắt ngang giữa epoch -- nhờ vậy luôn có checkpoint đầy đủ của epoch vừa xong, và cell đọc log/eval/dọn dẹp phía sau luôn chạy được để bạn biết kết quả ngay trong phiên đó. Phiên sau: add output phiên trước làm input, set `RESUME` (cell cấu hình phía trên) trỏ vào `checkpoint.pth`, `EPOCHS` giữ nguyên 15. Chỉ bật `FINAL_EVAL_SWEEP` ở phiên cuối cùng.

In [ ]:
n_gpu = torch.cuda.device_count()
assert n_gpu == 2, f"notebook này thiết kế cho 2 GPU, đang thấy {n_gpu}"

if RESUME:
    launch_args = f"--resume {RESUME}"
else:
    launch_args = f"--pretrain_model_path {PRETRAIN} --finetune_ignore time_ diffusion"

cmd = (
    f"torchrun --standalone --nproc_per_node={n_gpu} main.py "
    f"-c {CONFIG} --datasets {DATASETS_JSON} --output_dir {OUT_DIR} --num_workers {NUM_WORKERS} "
    f"--max_epochs_this_run {MAX_EPOCHS_THIS_RUN} "
    f"{launch_args} --options {opt_str}"
)
print(cmd)
!{cmd}

## 7. Đọc log

In [ ]:
log_path = f"{OUT_DIR}/log.txt"
if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            print(line.rstrip())
else:
    print("chưa có log.txt ở", log_path)

## 8. Eval, quét số bước sampling

Cũng chạy 2 GPU cho nhanh, dù eval không throughput-critical bằng train.

In [ ]:
if FINAL_EVAL_SWEEP:
    for s in (1, 3, 5, 10):
        eval_out = f"{WORK}/output/eval_s{s}"
        cmd = (
            f"torchrun --standalone --nproc_per_node={n_gpu} main.py "
            f"-c {CONFIG} --datasets {DATASETS_JSON} --output_dir {eval_out} --eval --num_workers {NUM_WORKERS} "
            f"--resume {OUT_DIR}/checkpoint_best_regular.pth "
            f"--options {opt_str} diff_sampling_timesteps={s}"
        )
        print(cmd)
        !{cmd}
else:
    print("FINAL_EVAL_SWEEP=False -- bỏ qua eval sweep (bật ở phiên cuối, sau khi đã train đủ EPOCHS)")

## 9. Dọn dẹp + nhắc báo cáo

In [ ]:
!du -sh {OUT_DIR}
!ls -la {OUT_DIR}
print(f"Báo cáo kết quả kèm: EPOCHS={EPOCHS}, BATCH_SIZE/GPU={BATCH_SIZE}, n_gpu={n_gpu}, "
      f"tổng batch={BATCH_SIZE * n_gpu} (quy ước CLAUDE.md: luôn ghi iteration + batch size).")